# CMX on PST900: fast T4 version (300 epochs in about 20 minutes)

The full CMX setting (MiT-B2, 480×640 crops) needs about **78 GFLOPs per image** and takes many hours for 300 epochs on a T4.
This notebook uses a **fast setting** that keeps the CMX architecture but is about **24× cheaper per image**:

| | Full CMX setting | This notebook |
|---|---|---|
| Backbone | MiT-B2 (66.6 M params) | **MiT-B0** (11.2 M params), same CMX fusion (FRM + FFM) |
| Training crops | 480×640 from 720×1280 | **240×320 from 360×640** (half resolution) |
| Data loading | CPU, every epoch | **whole dataset cached on the GPU once**; augmentation on the GPU |
| Precision | fp32 | **mixed precision (fp16)** |
| Compute per image | 78.3 GFLOPs | **3.3 GFLOPs** |

Evaluation is always scored on the **original 720×1280 labels**.

Use the **same setting for every experiment (E1–E4)** so the comparisons stay fair.
Absolute numbers will be lower than the paper's full-resolution MiT-B2 results.

**Steps:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Check the paths in cell 1.
3. `Run all`.

Cell 4 measures the real speed first and prints the expected training time before it starts.

## 1. Settings

In [ ]:
# --- files (Google Drive paths) ---
DATASET     = '/content/drive/MyDrive/AIProj_CMX/datasets/PST900_RGBT_Dataset.zip'  # zip or extracted folder
WEIGHTS_DIR = '/content/drive/MyDrive/AIProj_CMX/pretrained'    # mit_b0.pth goes here (auto-downloaded if missing)
OUT_DIR     = '/content/drive/MyDrive/AIProj_CMX/fast_runs'     # results + checkpoints

# --- experiment ---
EXP        = 'E1_cmx_b0_fast'
BACKBONE   = 'mit_b0'        # 'mit_b0' = fast (default); 'mit_b2' = ~5x slower
EPOCHS     = 300
BATCH_SIZE = 16
LR         = 6e-5 * BATCH_SIZE / 8   # repo LR (6e-5 at batch 8), scaled linearly with batch size
BASE_SIZE  = (360, 640)      # images are stored at half resolution (H, W)
CROP       = (240, 320)      # training crop (H, W)
SCALES     = (0.75, 1.75)    # random scale range (as in the repo, within what fits the crop)
VAL_EVERY  = 50              # quick test-set mIoU every N epochs (0 = only at the end)
TIME_BUDGET_MIN = None       # e.g. 20: if 300 epochs won't fit, EPOCHS is reduced to fit (else just warns)
SEED       = 0

## 2. Setup: Drive, packages, CMX code, pretrained weights

In [ ]:
import os, sys, glob, time, math, shutil, zipfile, subprocess
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

IN_COLAB = os.path.exists('/content')
if IN_COLAB and DATASET.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEV.type == 'cuda', 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = True
torch.manual_seed(SEED); np.random.seed(SEED)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'easydict', 'timm', 'opencv-python-headless', 'gdown', 'pandas'])
import cv2

WORK = '/content/work' if IN_COLAB else os.path.abspath('work')
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(WEIGHTS_DIR, exist_ok=True)
REPO = os.path.join(WORK, 'RGBX_Semantic_Segmentation')
if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', '-q', 'https://github.com/huaaaliu/RGBX_Semantic_Segmentation.git', REPO])
sys.path.insert(0, REPO)

# ImageNet-pretrained MiT weights (official CMX / SegFormer folder)
W_PATH = os.path.join(WEIGHTS_DIR, f'{BACKBONE}.pth')
if not os.path.exists(W_PATH):
    import gdown
    try:
        gdown.download_folder(url='https://drive.google.com/drive/folders/10XgSW8f7ghRs9fJ0dE-EV8G2E_guVsT5',
                              output=os.path.join(WORK, 'pretrained_dl'), quiet=True)
    except Exception as e:
        print('Automatic download failed:', e)
    found = glob.glob(os.path.join(WORK, 'pretrained_dl', '**', f'{BACKBONE}.pth'), recursive=True)
    if found:
        shutil.copy(found[0], W_PATH)
assert os.path.exists(W_PATH), (f'{BACKBONE}.pth not found. Download it from '
    'https://drive.google.com/drive/folders/10XgSW8f7ghRs9fJ0dE-EV8G2E_guVsT5 and put it in ' + WEIGHTS_DIR)
print('Pretrained weights:', W_PATH)

## 3. Load PST900 once into GPU memory
* Images are resized to 360×640 (half resolution).
* Test labels are also kept at the original 720×1280 for scoring.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
CLASSES = ['Background', 'Fire-Extinguisher', 'Backpack', 'Hand-Drill', 'Survivor']
NC = len(CLASSES)

src = DATASET
if src.endswith('.zip'):
    out = os.path.join(WORK, 'dataset')
    if not os.path.exists(out):
        local_zip = os.path.join(WORK, 'pst900.zip')
        print('Copying zip to local disk...'); shutil.copy(src, local_zip)
        print('Unzipping...'); zipfile.ZipFile(local_zip).extractall(out); os.remove(local_zip)
    src = out
ROOT = next(r for r, d, _ in os.walk(src) if 'train' in d and 'test' in d)

def read_sample(args):
    split, name = args
    rgb = cv2.cvtColor(cv2.imread(os.path.join(ROOT, split, 'rgb', name), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    th = cv2.imread(os.path.join(ROOT, split, 'thermal', name), cv2.IMREAD_GRAYSCALE)
    lab = cv2.imread(os.path.join(ROOT, split, 'labels', name), cv2.IMREAD_GRAYSCALE)
    lab = np.where(lab < NC, lab, 255).astype(np.uint8)          # anything else = ignore
    H, W = BASE_SIZE
    return (cv2.resize(rgb, (W, H), interpolation=cv2.INTER_AREA),
            cv2.resize(th, (W, H), interpolation=cv2.INTER_AREA),
            cv2.resize(lab, (W, H), interpolation=cv2.INTER_NEAREST), lab)

def load_split(split):
    names = sorted(os.path.basename(f) for f in glob.glob(os.path.join(ROOT, split, 'rgb', '*.png')))
    with ThreadPoolExecutor(8) as ex:
        data = list(ex.map(read_sample, [(split, n) for n in names]))
    rgb = torch.from_numpy(np.stack([d[0] for d in data])).permute(0, 3, 1, 2).contiguous()
    th = torch.from_numpy(np.stack([d[1] for d in data]))[:, None]
    lab = torch.from_numpy(np.stack([d[2] for d in data]))
    full = torch.from_numpy(np.stack([d[3] for d in data]))
    return rgb.to(DEV), th.to(DEV), lab.to(DEV), full

t0 = time.time()
tr_rgb, tr_th, tr_lab, _ = load_split('train')
te_rgb, te_th, _, te_full = load_split('test')
te_full = te_full.to(DEV)
print(f'train {len(tr_rgb)}  test {len(te_rgb)}  (PST900: 597 / 288)   loaded in {time.time()-t0:.0f}s')
print('label values:', torch.unique(tr_lab).tolist(), '| GPU memory used:', f'{torch.cuda.memory_allocated()/1e9:.1f} GB')

## 4. Model, GPU augmentation, speed check

In [ ]:
from types import SimpleNamespace
from models.builder import EncoderDecoder
from utils.init_func import group_weight

AMP_DTYPE = torch.float16 if DEV.type == 'cuda' else torch.bfloat16   # fp16 on the T4
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
def norm(x_uint8):                       # uint8 B x C x H x W -> normalized float
    x = x_uint8.float() / 255.0
    if x.shape[1] == 1: x = x.expand(-1, 3, -1, -1)   # thermal -> 3 channels
    return (x - MEAN) / STD

def gpu_augment(idx):
    # Random scale + random crop + horizontal flip for a batch, all on the GPU
    rgb, th, lab = tr_rgb[idx], tr_th[idx], tr_lab[idx]
    B, H, W = len(idx), *BASE_SIZE
    ch, cw = CROP
    s = torch.empty(B, device=DEV).uniform_(*SCALES)
    sy, sx = (ch / s) / H, (cw / s) / W                       # window size (normalized, <= 1)
    ty = (torch.rand(B, device=DEV) * 2 - 1) * (1 - sy)
    tx = (torch.rand(B, device=DEV) * 2 - 1) * (1 - sx)
    flip = torch.where(torch.rand(B, device=DEV) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=DEV)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = sx * flip, tx, sy, ty
    grid = F.affine_grid(theta, (B, 1, ch, cw), align_corners=False)
    x_rgb = F.grid_sample(norm(rgb), grid, mode='bilinear', align_corners=False)
    x_th = F.grid_sample(norm(th), grid, mode='bilinear', align_corners=False)
    y = F.grid_sample(lab[:, None].float(), grid, mode='nearest', align_corners=False)[:, 0].long()
    return x_rgb, x_th, y

def build_model():
    cfg = SimpleNamespace(backbone=BACKBONE, pretrained_model=W_PATH, decoder='MLPDecoder',
                          decoder_embed_dim=512 if BACKBONE == 'mit_b2' else 256,
                          num_classes=NC, bn_eps=1e-3, bn_momentum=0.1)
    m = EncoderDecoder(cfg=cfg, criterion=nn.CrossEntropyLoss(ignore_index=255), norm_layer=nn.BatchNorm2d)
    return m.to(DEV)

def make_optimizer(m):
    return torch.optim.AdamW(group_weight([], m, nn.BatchNorm2d, LR), lr=LR, betas=(0.9, 0.999), weight_decay=0.01)

# check that ImageNet weights really loaded (the repo loads with strict=False)
raw = torch.load(W_PATH, map_location='cpu', weights_only=False); raw = raw.get('model', raw)
m_chk = build_model(); own = m_chk.backbone.state_dict()
hits = sum(1 for k, v in raw.items() if k in own and torch.equal(own[k].cpu(), v))
print(f'pretrained tensors loaded: {hits}/{len(raw)}'); assert hits > 0.9 * len(raw)
print(f'params: {sum(p.numel() for p in m_chk.parameters())/1e6:.1f} M')

# speed check (on a throw-away copy)
crit = nn.CrossEntropyLoss(ignore_index=255)
scaler = torch.amp.GradScaler('cuda', enabled=DEV.type == 'cuda')
opt_chk = make_optimizer(m_chk); m_chk.train()
N = len(tr_rgb); IPE = N // BATCH_SIZE
def step(m, opt, idx):
    x_rgb, x_th, y = gpu_augment(idx)
    with torch.autocast(DEV.type, dtype=AMP_DTYPE):
        loss = crit(m(x_rgb, x_th), y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return loss
for k in range(25):
    if k == 5: torch.cuda.synchronize(); t0 = time.time()
    step(m_chk, opt_chk, torch.randint(0, N, (BATCH_SIZE,), device=DEV))
torch.cuda.synchronize()
sec_it = (time.time() - t0) / 20
eta_min = sec_it * IPE * EPOCHS / 60
print(f'{sec_it*1000:.0f} ms/iter | {IPE} iters/epoch | {sec_it*IPE:.1f} s/epoch | {EPOCHS} epochs ≈ {eta_min:.0f} min')
print(f'peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
if TIME_BUDGET_MIN and eta_min > TIME_BUDGET_MIN:
    EPOCHS = max(1, int(TIME_BUDGET_MIN * 60 / (sec_it * IPE)))
    print(f'>>> Reduced EPOCHS to {EPOCHS} to fit {TIME_BUDGET_MIN} min. Use the same EPOCHS for all experiments.')
del m_chk, opt_chk; torch.cuda.empty_cache()

## 5. Train

In [ ]:
import pandas as pd

@torch.no_grad()
def evaluate(m, bs=8):
    m.eval(); cm = torch.zeros(NC, NC, dtype=torch.long, device=DEV)
    for i in range(0, len(te_rgb), bs):
        with torch.autocast(DEV.type, dtype=AMP_DTYPE):
            out = m(norm(te_rgb[i:i+bs]), norm(te_th[i:i+bs]))
        pred = F.interpolate(out.float(), size=te_full.shape[-2:], mode='bilinear', align_corners=False).argmax(1)
        gt = te_full[i:i+bs].long(); keep = gt != 255
        cm += torch.bincount(gt[keep] * NC + pred[keep], minlength=NC * NC).view(NC, NC)
    m.train()
    return cm.cpu().double().numpy()

def miou(cm):
    tp = np.diag(cm); return float(np.nanmean(tp / (cm.sum(0) + cm.sum(1) - tp)))

model = build_model(); opt = make_optimizer(model); scaler = torch.amp.GradScaler('cuda', enabled=DEV.type == 'cuda')
TOTAL = EPOCHS * IPE; WARM = min(10 * IPE, TOTAL // 20)       # repo: poly decay (power 0.9) + linear warm-up
def lr_at(it):
    return LR * it / WARM if it < WARM else LR * (1 - it / TOTAL) ** 0.9

CKPT = os.path.join(OUT_DIR, f'{EXP}_ckpt.pth'); start, log = 0, []
if os.path.exists(CKPT):                                     # resume after a disconnect
    ck = torch.load(CKPT, map_location='cpu', weights_only=False)
    if ck['epochs'] == EPOCHS:
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
        start, log = ck['epoch'], ck['log']; print('Resumed from epoch', start)

model.train(); t_start = time.time()
for ep in range(start, EPOCHS):
    perm = torch.randperm(N, device=DEV); losses = []
    for b in range(IPE):
        it = ep * IPE + b
        for g in opt.param_groups: g['lr'] = lr_at(it)
        loss = step(model, opt, perm[b*BATCH_SIZE:(b+1)*BATCH_SIZE])
        if b % 10 == 0: losses.append(loss.item())          # sync rarely (faster)
    row = {'epoch': ep + 1, 'loss': float(np.mean(losses)), 'lr': lr_at(it), 'minutes': (time.time() - t_start) / 60}
    if VAL_EVERY and ((ep + 1) % VAL_EVERY == 0 or ep + 1 == EPOCHS):
        row['test_mIoU'] = miou(evaluate(model)) * 100
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': ep + 1, 'epochs': EPOCHS, 'log': log + [row]}, CKPT)
    log.append(row)
    if (ep + 1) % 10 == 0 or 'test_mIoU' in row:
        el = (time.time() - t_start) / 60; left = el / (ep + 1 - start) * (EPOCHS - ep - 1)
        print(f"epoch {ep+1}/{EPOCHS} | loss {row['loss']:.3f} | {el:.1f} min elapsed, ~{left:.1f} min left"
              + (f" | test mIoU {row['test_mIoU']:.2f}%" if 'test_mIoU' in row else ''))
print(f'Training done in {(time.time() - t_start)/60:.1f} min')
pd.DataFrame(log).to_csv(os.path.join(OUT_DIR, f'{EXP}_train_log.csv'), index=False)

## 6. Final evaluation (original 720×1280 labels)

In [ ]:
cm = evaluate(model)
tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
res = pd.DataFrame({'IoU': tp / (tp + fp + fn), 'Dice': 2 * tp / (2 * tp + fp + fn),
                    'Sensitivity': tp / (tp + fn), 'Specificity': tn / (tn + fp)}, index=CLASSES) * 100
res.loc['Mean'] = res.iloc[:NC].mean()
res.loc['Mean (objects, no bg)'] = res.iloc[1:NC].mean()
res = res.round(2)
res.to_csv(os.path.join(OUT_DIR, f'{EXP}_results.csv'))
torch.save({'model': model.state_dict(), 'backbone': BACKBONE, 'epochs': EPOCHS},
           os.path.join(OUT_DIR, f'{EXP}_final_weights.pth'))
print(f"{EXP}: mIoU {res.loc['Mean', 'IoU']:.2f}% | {EPOCHS} epochs | saved to {OUT_DIR}")
res